<a href="https://colab.research.google.com/github/chrisjmccormick/shared-subspaces/blob/main/subspace_decoder/scripts/run_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ▂▂▂▂▂▂▂▂▂▂▂▂

# Overview

This notebook demonstrates how to run the pre-training and fine-tuning scripts from a command line, and is also setup to allow you to run then from within the notebook.

**Running on Colab**

The current configurations require the 40GB A100 because of the pre-training batch size.

(If you wanted to run on a T4, you could adjust the training arguments to use batch accumulation. This would allow you to preserve the training behavior without running out of memory.)

What to expect:

* Pre-training runs take roughly 75 minutes. It's pushing the limit of what works well in Colab--you'll want to babysit the notebook a little to avoid disconnect issues.

* Fine-tuning is relatively fast, and completes in under 10 minutes.

**Training Arguments**

I designed the scripts such that everything is specified through `.json` config files rather than on the command line. However, there is a command line utility to define new configurations--see the "Defining a New Run" section at the end of this notebook.

The examples below run one of the existing configurations.

**Weights and Biases**

The scripts are set up to log to wandb by default. You can change the `wandb_mode` variable below to 'offline' if you don't have an account / don't want to log online.

The project names are currently hardcoded to:

* Pretraining: `ethos-pretrain-wiki103`



# ▂▂▂▂▂▂▂▂▂▂▂▂

# S1. Setup

In [ ]:
# Wes' installation steps:
!pip3 install -U torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
!pip3 install datasets tiktoken transformers tqdm deepspeed wandb matplotlib pandas numpy

In [ ]:
import importlib
import sys
import os
import textwrap

def print_env_info():
    # Packages you care about
    packages = [
        "torch",
        "torchvision",
        "transformers",
        "datasets",
        "accelerate",
        "flash_attn",
        "wandb",
        "numpy",
        "pandas",
    ]
    
    # Print Python version first
    print(f"{'Python':<15} {sys.version.split()[0]:<15} {sys.executable}")
    print("-" * 80)
    
    for pkg in packages:
        try:
            module = importlib.import_module(pkg)
            version = getattr(module, "__version__", "unknown")
            location = os.path.dirname(module.__file__)
            print(f"{pkg:<15} {version:<15} {location}")
        except ModuleNotFoundError:
            print(f"{pkg:<15} {'(not installed)':<15}")
    
    # CUDA info if torch is present
    try:
        import torch
        print("-" * 80)
        print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"cuDNN version: {torch.backends.cudnn.version()}")
    except Exception:
        pass

print_env_info()


In [ ]:
#!pip install wandb datasets transformers tf-keras accelerate

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 36.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 KB 125.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 97.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 145.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 KB 109.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.2/208.2 KB 73.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 KB 85.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.3/368.3 KB 120.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 KB 157.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 KB 76.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 KB 60.8 MB/s eta 0:00:00
     ━━━━━━━

## 1.1. Clone Repository

In [2]:
# Clone the "ethos" branch
!git clone https://github.com/chrisjmccormick/shared-subspaces.git
!cd shared-subspaces && git checkout ethos

Cloning into 'shared-subspaces'...
remote: Enumerating objects: 273, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 273 (delta 52), reused 48 (delta 48), pack-reused 200 (from 1)
Receiving objects: 100% (273/273), 10.86 MiB | 74.17 MiB/s, done.
Resolving deltas: 100% (139/139), done.
Branch 'ethos' set up to track remote branch 'ethos' from 'origin'.
Switched to a new branch 'ethos'


In [3]:
%cd shared-subspaces/
!git pull

/home/ubuntu/shared-subspaces
Already up to date.


Provide the full path to the subspace_decoder folder.

This will be added to the PYTHONPATH when executing the scripts so that they can import the classes from the local files.

This variable is also used to construct paths to config files and scripts.

In [5]:
%cd ..
!ls

/home/ubuntu
run_ethos_experiments.ipynb  shared-subspaces


In [1]:
base_path = "/home/ubuntu/shared-subspaces/ethos"

In [2]:

import sys, numpy as np
print("NumPy:", np.__version__, np.__file__)
for m in ("torch","pandas"):
    try:
        mod = __import__(m)
        print(f"{m}:", getattr(mod,"__version__", "n/a"), mod.__file__)
    except Exception as e:
        print(f"{m}: import error -> {e}")
print("sys.path[0:5]:", sys.path[:5])




NumPy: 2.2.6 /home/ubuntu/.local/lib/python3.10/site-packages/numpy/__init__.py



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/lib/python3/dist-packages/ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "/usr/lib/python3/dist-packages/traitlets/config/application.py", line 846, in launch_instance
    app.start()
  File "/usr/lib/python3/dist-packages/ipykernel/kernelapp.py", line 677, in start
    s

torch: 2.7.0 /usr/lib/python3/dist-packages/torch/__init__.py
pandas: import error -> numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject
sys.path[0:5]: ['/home/ubuntu', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '']


## 1.2. Weights & Biases

To provide your wandb API key for the script:
1. You could paste it in manually on the training command lines further down.
2. Or, use the secrets panel (the key symbol on the left edge of the notebook) and:
    * Define your wandb api key as `wandb_api_key`.
    * Grant access to this notebook.
    * Run the below cell to retrieve it.

In [3]:
# Set to false if you don't want to use wandb.
# The scripts will still log using the wandb library, but to a local directory.
use_wandb = True

if use_wandb:
    # Enable Weights & Biases logging (online mode)
    wandb_mode = "online"
    wandb_key = "d63437856880292b3bc7e96dd31816c4c48af4b9"

# Set to offline if you don't want to log in.
else:
    wandb_mode = "offline"

    wandb_key = ""

## 1.3. Choose Configuration

In [4]:
import os
import json

#  Choose which config file to run
pretrain_config_path = f"{base_path}/configs/ethos_baseline.json"
#pretrain_config_path = f"{base_path}/configs/best_mla-o.json"

# Make sure it's a valid path
if not os.path.exists(pretrain_config_path):
    raise ValueError(f"Config file {pretrain_config_path} does not exist.")

# Print it out.
with open(pretrain_config_path, "r") as f:
    pretrain_config = json.load(f)

print(f"\n======== {pretrain_config_path} ========\n")

# Print out the configuration with spacing.
json_str = json.dumps(pretrain_config, indent=4)
print(json_str)


======== /home/ubuntu/shared-subspaces/ethos/configs/ethos_baseline.json ========

{
    "shorthand": "seqlen.128 - mla.96.64 - mlp.1024 - model.256.lyr.6 - ah.8.32",
    "notes": "Baseline ETHOS model.",
    "model": {
        "hidden_size": 256,
        "num_hidden_layers": 6,
        "intermediate_size": 1024,
        "num_moe_layers": 5,
        "num_dense_layers": 1,
        "num_experts": 65536,
        "d_latent": 32,
        "d_intermediate_hypernet": 128,
        "top_k": 8,
        "num_routing_heads": 4,
        "d_query": 32,
        "pad_token_id": 50256,
        "bos_token_id": 50256,
        "eos_token_id": 50256,
        "tie_word_embeddings": true,
        "attention_dropout": 0.0,
        "hidden_dropout_prob": 0.1,
        "classifier_dropout": null,
        "initializer_range": 0.02,
        "rms_norm_eps": 1e-06,
        "vocab_size": 50257,
        "rope_theta": 10000.0,
        "rope_scaling": null,
        "max_seq_len": 128,
        "kv_lora_rank": 64,
       

# S2. Run Pre-Training

In [8]:
print("\n======= Pre-Train ========\n")

# Construct the command line
train_command = (
    f"PYTHONPATH={base_path} "
    f"WANDB_MODE={wandb_mode} "
    f'WANDB_API_KEY="{wandb_key}" '
    f"python {base_path}/scripts/train.py --config {pretrain_config_path}"
)

# Run pre-training
!{train_command}


======= Pre-Train ========

Importing Packages...

Traceback (most recent call last):
  File "/home/ubuntu/shared-subspaces/ethos/scripts/train.py", line 18, in <module>
    from transformers import (
  File "<frozen importlib._bootstrap>", line 1075, in _handle_fromlist
  File "/home/ubuntu/.local/lib/python3.10/site-packages/transformers/utils/import_utils.py", line 2302, in __getattr__
    module = self._get_module(self._class_to_module[name])
  File "/home/ubuntu/.local/lib/python3.10/site-packages/transformers/utils/import_utils.py", line 2332, in _get_module
    raise e
  File "/home/ubuntu/.local/lib/python3.10/site-packages/transformers/utils/import_utils.py", line 2330, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
  File "/usr/lib/python3.10/importlib/__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/transformers/trainer.py", l

In [9]:
print(train_command)

PYTHONPATH=/home/ubuntu/shared-subspaces/ethos WANDB_MODE=online WANDB_API_KEY="d63437856880292b3bc7e96dd31816c4c48af4b9" python /home/ubuntu/shared-subspaces/ethos/scripts/train.py --config /home/ubuntu/shared-subspaces/ethos/configs/ethos_baseline.json


**Optional - Save the checkpoint to Google Drive**

In [ ]:
import shutil

if False:
    # Copy whatever's in the checkpoints folder over to Google Drive.
    shutil.copytree(
        "/content/checkpoints",
        "/content/drive/MyDrive/ethos-pretrain/checkpoints",
        dirs_exist_ok = True
    )

'/content/drive/MyDrive/encoder-pretrain-wiki103/checkpoints'

# ▂▂▂▂▂▂▂▂▂▂▂▂

# Defining a New Run

To modify the parameters from the command line, I created a command line utiltity in `/configs/create_new_config.py` which will copy one of the existing config files and allow you to specify any parameter changes.

See the [script](https://github.com/chrisjmccormick/shared-subspaces/blob/main/subspace_encoder/configs/create_new_config.py) for documentation, check out the baseline config [here](https://github.com/chrisjmccormick/shared-subspaces/blob/main/subspace_encoder/configs/best_mla-o.json) to see all of the hyperparameters that are defined, and see the [Config](https://github.com/chrisjmccormick/shared-subspaces/blob/main/subspace_encoder/models/shared_space_config.py#L81) class for documentation of the model parameters.

Below is an example for defining a new run which increases the output latent size to 96.

In [ ]:
!python {base_path}/configs/create_new_config.py \
    mla-o_baseline_o96
    --base {base_path}/configs/mla-o_baseline.json \
    --shorthand "rd.32 - 6.mla.64.32.96 - mlp.1024 - model.256.lyr.6 - ah.8.32" \
    --notes "Trying increasing the output subspace size from 64 to 96" \
    --set model.o_latent_dim=96

Wrote new config to /content/shared-subspaces/subspace_encoder/configs/mla-o_baseline_o96.json


In [ ]:
!cat {base_path}/configs/mla-o_baseline_o96.json

{
  "shorthand": "rd.32 - 6.mla.64.32.96 - mlp.1024 - model.256.lyr.6 - ah.8.32",
  "notes": "Trying increasing the output subspace size from 64 to 96",
  "model": {
    "hidden_size": 256,
    "num_hidden_layers": 6,
    "intermediate_size": 1024,
    "hidden_dropout_prob": 0.1,
    "attention_dropout_prob": 0.1,
    "classifier_dropout": null,
    "initializer_range": 0.02,
    "layer_norm_eps": 1e-12,
    "rms_norm_eps": 1e-06,
    "vocab_size": 30522,
    "rope_theta": 10000.0,
    "rope_scaling": null,
    "max_position_embeddings": 128,
    "num_dense_layers": 0,
    "q_latent_dim": 64,
    "kv_latent_dim": 32,
    "num_attention_heads": 8,
    "head_dim": 32,
    "rope_dims": 32,
    "attention_bias": false,
    "output_subspace": true,
    "o_latent_dim": 96,
    "attention_backend": "sdpa",
    "ffn_decompose": false,
    "ffn_rank": null,
    "vocab_subspace": false,
    "vocab_rank": 128
  },
  "pre_train": {
    "output_dir": "checkpoints/mla-o_baseline_o96",
    "seed": 42

# ▂▂▂▂▂▂▂▂▂▂▂▂

In [6]:
# -*- coding: utf-8 -*-
# Updated training script for DeepSeek V3 with attention output subspace

"""# subspace_decoder/scripts/train.py"""

print("Importing Packages...\n")

import argparse
import json
import os
import shutil
import sys
from pathlib import Path

import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    set_seed,
)

from utils import summarize_parameters, format_size
# To disable a warning.
os.environ["TOKENIZERS_PARALLELISM"] = "false"


class RouterStatsCallback(TrainerCallback):
    """Callback to log MoE router statistics during training."""
    
    def __init__(self, log_every_n_steps=100):
        self.log_every_n_steps = log_every_n_steps
    
    def on_step_end(self, args, state, control, model=None, **kwargs):
        """Called at the end of each training step."""
        if state.global_step % self.log_every_n_steps == 0:
            try:
                # Access the transformer layers (handle both wrapped and unwrapped models)
                if hasattr(model, 'model') and hasattr(model.model, 'layers'):
                    layers = model.model.layers  # EthosForCausalLM case
                elif hasattr(model, 'layers'):
                    layers = model.layers  # Direct EthosModel case
                else:
                    print(f"Warning: Cannot find model layers for router stats logging")
                    return
                
                # Log each router separately.
                router_stats_logged = False
                for i, layer in enumerate(layers):
                    # If it's an moe layer,
                    if hasattr(layer.mlp, "router"):
                        stats = layer.mlp.router.consume_stats()
                        # Log to wandb with the current step
                        wandb.log({f"moe_layer{i}/{k}": v for k, v in stats.items()}, step=state.global_step)
                        router_stats_logged = True
                
                if not router_stats_logged:
                    print(f"Warning: No MoE layers with routers found at step {state.global_step}")
                    
            except Exception as e:
                print(f"Error logging router stats at step {state.global_step}: {e}")

# Make sure we can import modules from the decoder package
PROJECT_ROOT = Path(__file__).resolve().parents[1]

print("PROJECT_ROOT", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import our custom ETHOS model
from models.configuration_ethos import EthosConfig
from models.ethos import EthosForCausalLM

import torch.nn as nn


def check_bf16_support():
    """Check if BFloat16 is supported on the current hardware and PyTorch version."""
    if not torch.cuda.is_available():
        print("Warning: CUDA not available. BFloat16 training requires CUDA.")
        return False
    
    # Check if the GPU supports BFloat16
    if hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported():
        print("✓ BFloat16 is supported on this hardware")
        return True
    
    # Fallback check for older PyTorch versions
    try:
        # Try to create a small BFloat16 tensor on GPU
        test_tensor = torch.tensor([1.0], dtype=torch.bfloat16, device='cuda')
        print("✓ BFloat16 is supported on this hardware")
        return True
    except Exception as e:
        print(f"Warning: BFloat16 not supported on this hardware: {e}")
        return False

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, help="Path to JSON config")
    return parser.parse_args()


def main(config_path: str):
    """Run pre-training using the provided configuration path."""
    
    # Load configuration
    with open(config_path, 'r') as f:
        full_cfg = json.load(f)

    model_cfg = full_cfg['model']
    ptrain_cfg = full_cfg['pre_train']

    # Print out its shorthand name.
    print(full_cfg["shorthand"])

    # Initialize the optional stats dictionary so later assignments don't fail.
    if "stats" not in full_cfg:
        full_cfg["stats"] = {}
    
    # Validate mixed precision settings
    if ptrain_cfg.get("bf16", False) and ptrain_cfg.get("fp16", False):
        raise ValueError("Cannot enable both bf16 and fp16 simultaneously. Please choose one.")
    
    # Check BFloat16 compatibility if enabled
    if ptrain_cfg.get("bf16", False):
        if not check_bf16_support():
            print("BFloat16 requested but not supported. Falling back to FP16.")
            ptrain_cfg["bf16"] = False
            ptrain_cfg["fp16"] = True
    
    # Display torch.compile status
    if ptrain_cfg.get("torch_compile", False):
        print(f"✓ torch.compile enabled:")
        print(f"  Backend: {ptrain_cfg.get('torch_compile_backend', 'inductor')}")
        print(f"  Mode: {ptrain_cfg.get('torch_compile_mode', 'default')}")
        print("  Note: First training step will be slower due to compilation.")
    else:
        print("torch.compile disabled. Enable with 'torch_compile': true in config.")

    # Use the DeepSeek tokenizer
    #tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3")
    
    # Set pad token if not already set
    #if tokenizer.pad_token is None:
    #    tokenizer.pad_token = tokenizer.eos_token

    
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    # gpt2 has no pad by default; use EOS for padding in causal LM
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
        
    # Verify vocab size matches
    assert model_cfg["vocab_size"] == tokenizer.vocab_size

    # Set random seed for reproducibility
    set_seed(ptrain_cfg["seed"])

    # Setup Weights & Biases
    if "WANDB_MODE" not in os.environ:
        os.environ["WANDB_MODE"] = "offline"

    wandb_api_key = os.environ.get("WANDB_API_KEY")

    if wandb_api_key:
        wandb.login(key=wandb_api_key)

    # ======================
    #    Prepare Dataset
    # ======================
    
    dataset = load_dataset(
        ptrain_cfg["dataset_name"],
        ptrain_cfg["dataset_config"]
    )
    print(dataset)
    
    block_size = ptrain_cfg["max_seq_length"]
    eos_id = tokenizer.eos_token_id
    
    # 1) Tokenize without truncation/padding
    def tokenize_function(examples):
        # add_special_tokens=False keeps things raw; we’ll insert EOS between docs
        return tokenizer(
            examples["text"],
            add_special_tokens=False,
        )
    
    # 2) Group into contiguous blocks (concat + chunk)
    def group_texts(examples):
        # Flatten and insert EOS between documents to avoid cross-article bleed
        input_ids = []
        for ids in examples["input_ids"]:
            if len(ids) > 0:
                input_ids.extend(ids)
            # add an EOS fencepost between docs
            input_ids.append(eos_id)
    
        # Drop the trailing partial block so every example is full length
        total_length = (len(input_ids) // block_size) * block_size
        input_ids = input_ids[:total_length]
    
        # Split into equal blocks
        result_input_ids = [input_ids[i:i + block_size] for i in range(0, total_length, block_size)]
        # Labels are next-token targets; Trainer/model will do the shift
        return {
            "input_ids": result_input_ids,
            "labels": [ids.copy() for ids in result_input_ids],
            # Optional attention masks (all ones because no padding)
            "attention_mask": [[1] * block_size for _ in result_input_ids],
        }
    
    # Tokenize
    tokenized = dataset.map(
        tokenize_function,
        batched=True,
        num_proc=8,
        remove_columns=dataset["train"].column_names,  # drop raw "text"
    )
    
    # Concatenate + chunk
    tokenized = tokenized.map(
        group_texts,
        batched=True,
        num_proc=8,
    )
    
    # Use a simple collator; we already created labels and have no pads
    from transformers import default_data_collator
    data_collator = default_data_collator



    # ========================
    #    Initialize Model
    # ========================

    print("Initializing model...")

    # Create ETHOS config from model config
    config = EthosConfig(**model_cfg)
    
    # Initialize the ETHOS model
    model = EthosForCausalLM(config)

    # ================================
    #       Review Configuration
    # ================================

    # Display architecture
    print(model)

    print("\n======== Model ========")
    print(json.dumps(model_cfg, indent=2))

    print("\n======== Pre-Train ========")
    print(json.dumps(ptrain_cfg, indent=2))

    # Calculate and display effective batch size
    device_batch_size = ptrain_cfg["train_batch_size"]
    gradient_accumulation_steps = ptrain_cfg.get("gradient_accumulation_steps", 1)
    effective_batch_size = device_batch_size * gradient_accumulation_steps
    
    print(f"\n======== Batch Size Configuration ========")
    print(f"Device batch size: {device_batch_size}")
    print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
    print(f"Effective batch size: {effective_batch_size}")

    print("=============================\n")

    """## Parameter Summary"""

    print("\n======== Parameters ========")

    ## Get all of the model's parameters as a list of tuples.
    params = list(model.named_parameters())

    print('The model has {:} different named parameters.\n'.format(len(params)))

    total_params = 0
    for p_name, p in params:
        total_params += p.numel()

    full_cfg["stats"]["total_elements"] = format_size(total_params)

    print(f"Total elements: {full_cfg['stats']['total_elements']}\n")

    # Display a full parameter breakdown using the shared utility
    summarize_parameters(model)

    # ========================================
    #   Format Settings for WandB Run Name
    # ========================================

    # Format the cfg learning rate as a scientific notation string like 5e-4
    lr_str = '{:.0e}'.format(ptrain_cfg['learning_rate'])

    # Attention configuration

    ptrain_cfg["run_name"] = full_cfg["stats"]["total_elements"] + " - " + full_cfg["shorthand"]

    print(ptrain_cfg["run_name"])

    """## wandb and TrainingArguments"""

    wandb.init(
        project="ethos-pretrain-wiki103",
        name=ptrain_cfg["run_name"],
        config=full_cfg
    )


    # ===============================
    #       Training Arguments
    # ===============================

    training_args = TrainingArguments(
        output_dir=ptrain_cfg["output_dir"],

        per_device_train_batch_size=ptrain_cfg["train_batch_size"],
        per_device_eval_batch_size=ptrain_cfg["eval_batch_size"],
        gradient_accumulation_steps=ptrain_cfg.get("gradient_accumulation_steps", 1),

        bf16=ptrain_cfg.get("bf16", False),
        fp16=ptrain_cfg.get("fp16", False),
        
        # torch.compile configuration for performance optimization
        torch_compile=ptrain_cfg.get("torch_compile", False),
        torch_compile_backend=ptrain_cfg.get("torch_compile_backend", "inductor"),
        torch_compile_mode=ptrain_cfg.get("torch_compile_mode", "default"),

        learning_rate=ptrain_cfg["learning_rate"],
        max_steps=ptrain_cfg["num_train_steps"], 

        # The dataloader is a bottleneck without these.
        dataloader_num_workers=ptrain_cfg.get("num_workers", 8),
        dataloader_pin_memory=ptrain_cfg.get("pin_memory", True),
        # The prefetch factor didn't appear to help.
        #dataloader_prefetch_factor = ptrain_cfg.get("prefetch_factor", 2),

        weight_decay=ptrain_cfg.get("weight_decay", 0.01),  

        # Learning rate warmup (10% of total steps)
        warmup_steps=int(0.1 * ptrain_cfg["num_train_steps"]),  
        lr_scheduler_type="linear",  # Linear warmup then decay

        # Evaluate every 2,000 steps
        # Note: Recent versions of Trainer changed the name from 
        # `evaluation_strategy` to `eval_strategy`.
        batch_eval_metrics = True, # To avoid OOM
        eval_strategy="steps",
        eval_steps=ptrain_cfg.get("eval_steps", 2000),
        eval_accumulation_steps=4,  # Process eval in smaller chunks to save memory

        logging_steps=50,
        metric_for_best_model="eval_loss",
        save_steps=2000,
        save_total_limit=2,           # Optional: keeps last 2 checkpoints
        save_strategy="steps",
        report_to=["wandb"],
        
        run_name=ptrain_cfg["run_name"],
        
        remove_unused_columns=False,  # Optional: avoid dropping custom model inputs
    )

    print(training_args)

    import numpy as np

    class PerplexityMetric:
        """
        A stateful class to compute perplexity in a batch-wise manner to avoid OOM.
        Similar to the MLMAccuracyMetric from the encoder training.
        """
        def __init__(self):
            # Initialize state variables to store running totals
            self.total_loss = 0.0
            self.total_tokens = 0

        def __call__(self, eval_pred, compute_result=False):
            """
            This method will be called by the Trainer.
            """
            predictions, labels = eval_pred

            # For causal LM, we compute perplexity
            # Shift predictions and labels for next token prediction
            shift_logits = predictions[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            
            # Flatten the tokens
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            
            # Create a mask for valid tokens (not padding, typically -100)
            mask = shift_labels != -100
            
            if mask.sum() > 0:  # Only compute if there are valid tokens
                # Compute loss only on valid tokens
                loss_fct = torch.nn.CrossEntropyLoss(reduction='sum')
                batch_loss = loss_fct(shift_logits[mask], shift_labels[mask])
                
                # Add to running totals
                self.total_loss += batch_loss.item()
                self.total_tokens += mask.sum().item()

            # If this is the final call after all batches are processed
            if compute_result:
                # Avoid division by zero
                if self.total_tokens == 0:
                    avg_loss = 0.0
                    perplexity = float('inf')
                else:
                    avg_loss = self.total_loss / self.total_tokens
                    perplexity = np.exp(avg_loss)

                # Prepare the final metrics dictionary
                metrics = {
                    "perplexity": perplexity,
                    "loss": avg_loss,
                }

                # Reset state for the next evaluation run
                self.total_loss = 0.0
                self.total_tokens = 0

                return metrics

            # For intermediate calls, return an empty dict
            return {}

    # Instantiate your stateful metric computer
    perplexity_metric = PerplexityMetric()

    # ===============================
    #           Trainer
    # ===============================
    # Create router stats callback
    router_stats_callback = RouterStatsCallback(log_every_n_steps=100)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        compute_metrics=perplexity_metric,

        # New argument, allows for other modalities.
        processing_class=tokenizer,

        data_collator=data_collator,
        callbacks=[router_stats_callback],
    )

    """## Loop"""

    # =====================
    #     Run Training
    # =====================

    # Do inside a try/finally so that if the run aborts, we still call wandb.finish().
    try:
        trainer.train()

        metrics = trainer.evaluate()

        wandb.log(metrics)

        # Store wandb ids into the config.
        full_cfg["pre_train"]["run_id"] = wandb.run.id
        full_cfg["pre_train"]["run_url"] = wandb.run.url
        full_cfg["pre_train"]["run_name"] = wandb.run.name

        # Save the best checkpoint.
        full_cfg["pre_train"]["best_checkpoint"] = trainer.state.best_model_checkpoint

        # Save the json back to disk
        with open(ptrain_cfg["output_dir"] + "/full_config.json", "w") as f:
            json.dump(full_cfg, f, indent=2)
   

    finally:
        # End the wandb run.
        wandb.finish()

    
if __name__ == "__main__":
    #args = parse_args()
    #main(args.config)
    main(pretrain_config_path)


Importing Packages...



ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject